# 00. Data fetching

The two fetch scripts under `scripts/00_data_fetching/` are run from the CLI to populate `data/`:

```bash
uv run python scripts/00_data_fetching/fetch_daily.py     # -> data/daily_alpha, data/daily_yf
uv run python scripts/00_data_fetching/fetch_options.py   # -> data/options_alpha
```

Both iterate `config.UNIVERSE_TICKERS` (65-ticker superset). This notebook loads the resulting CSVs and sanity-checks coverage against `config.FILTERED_TICKERS` (59 tickers retained after filtering).


In [9]:
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from config import FILTERED_TICKERS

## Filtered ticker list

`FILTERED_TICKERS` drops 6 symbols from the initial watchlist: EXOD, KOSS, QBTQF, QMCO, QNTM (no options data) and CCCX (symbol changed to INFQ).


In [10]:
print(f"Loaded {len(FILTERED_TICKERS)} tickers from config.")

Loaded 59 tickers from config.


In [11]:
FILTERED_TICKERS

['AAPL',
 'ALKT',
 'AMAT',
 'AMD',
 'AMZN',
 'APLD',
 'ARQQ',
 'AVGO',
 'BBAI',
 'BE',
 'BULL',
 'BYND',
 'BYRN',
 'CAI',
 'CEG',
 'CLOV',
 'COIN',
 'DDD',
 'DELL',
 'DNUT',
 'GME',
 'GOOGL',
 'INTC',
 'IONQ',
 'KOPN',
 'KULR',
 'LAC',
 'LAES',
 'LRCX',
 'LTBR',
 'META',
 'MSFT',
 'NFE',
 'NFLX',
 'NNE',
 'NVDA',
 'NVTS',
 'OKLO',
 'PLTR',
 'QBTS',
 'QCOM',
 'QS',
 'QSI',
 'QUBT',
 'RBLX',
 'RGTI',
 'RKLB',
 'RR',
 'SERV',
 'SLI',
 'SMCI',
 'SMR',
 'SOUN',
 'TLRY',
 'TSLA',
 'TSM',
 'UMAC',
 'UUUU',
 'ZETA']

## Daily OHLCV from AlphaVantage

Sample: AAPL. Covers full history back to 1999 and runs through `settings.UNIVERSE_END_DATE` (2026-04-02).


In [12]:
av = pd.read_csv("data/daily_alpha/AAPL.csv")
av["date"] = pd.to_datetime(av["date"])
print(
    f"rows: {len(av)}  |  first: {av['date'].min().date()}  |  last: {av['date'].max().date()}"
)
av.tail()

rows: 6645  |  first: 1999-11-01  |  last: 2026-04-02


,date,open,high,low,close,adjusted_close,volume,dividend_amount,split_coefficient,source
6640,2026-03-27,253.90,255.493,248.070,248.80,248.80,47899998,0.0,1.0,alphavantage
6641,2026-03-30,250.07,250.870,245.510,246.63,246.63,39446213,0.0,1.0,alphavantage
6642,2026-03-31,247.91,255.480,247.101,253.79,253.79,49598091,0.0,1.0,alphavantage
6643,2026-04-01,254.08,256.180,253.330,255.63,255.63,40059432,0.0,1.0,alphavantage
6644,2026-04-02,254.20,256.130,250.650,255.92,255.92,31289369,0.0,1.0,alphavantage


## Daily OHLCV from yfinance

Sample: AAPL. yfinance reaches further back (1980) and may extend a few trading days past the AlphaVantage cutoff since it refreshes live. Cap to the universe end date for an apples-to-apples comparison.


In [13]:
yf_df = pd.read_csv("data/daily_yf/AAPL.csv")
yf_df["date"] = pd.to_datetime(yf_df["date"])
print(
    f"rows: {len(yf_df)}  |  first: {yf_df['date'].min().date()}  |  last: {yf_df['date'].max().date()}"
)
yf_df[yf_df["date"] <= "2026-04-02"].tail()

rows: 11419  |  first: 1980-12-12  |  last: 2026-04-06


,date,open,high,low,close,adjusted_close,volume,dividend_amount,split_coefficient,source
11413,2026-03-27,253.899994,255.490005,248.070007,248.800003,248.800003,47900000,0.0,1.0,yfinance
11414,2026-03-30,250.070007,250.869995,245.509995,246.630005,246.630005,39446200,0.0,1.0,yfinance
11415,2026-03-31,247.910004,255.479996,247.100006,253.789993,253.789993,49598100,0.0,1.0,yfinance
11416,2026-04-01,254.080002,256.179993,253.330002,255.630005,255.630005,40059400,0.0,1.0,yfinance
11417,2026-04-02,254.199997,256.130005,250.649994,255.919998,255.919998,31289400,0.0,1.0,yfinance


## File coverage

Both daily directories should have 65 CSVs and `options_alpha/` should have 65 subfolders. Five of the options subfolders will be empty (EXOD, KOSS, QBTQF, QMCO, QNTM) since those tickers do not trade options.


In [14]:
av_files = sorted(p.stem for p in Path("data/daily_alpha").glob("*.csv"))
yf_files = sorted(p.stem for p in Path("data/daily_yf").glob("*.csv"))
opt_dirs = sorted(p.name for p in Path("data/options_alpha").iterdir() if p.is_dir())
print(f"daily_alpha  : {len(av_files)} files")
print(f"daily_yf     : {len(yf_files)} files")
print(f"options_alpha: {len(opt_dirs)} tickers")

daily_alpha  : 65 files
daily_yf     : 65 files
options_alpha: 65 tickers


## Options chain sample

Each options file is the full chain for a (ticker, trade_date) pair: contract ID, strike, type, quote, open interest, and greeks.


In [15]:
opt = pd.read_csv("data/options_alpha/AAPL/AAPL_options_2026-04-02.csv")
print(f"shape: {opt.shape}")
opt.head()

shape: (3222, 20)


,contractID,symbol,expiration,strike,type,last,mark,bid,bid_size,ask,ask_size,volume,open_interest,date,implied_volatility,delta,gamma,theta,vega,rho
0,AAPL260402C00110000,AAPL,2026-04-02,110.0,call,142.80,145.70,144.00,167,147.40,165,0,1,2026-04-02,0.01488,1.00000,0.00000,-0.01097,0.00000,0.00301
1,AAPL260402P00110000,AAPL,2026-04-02,110.0,put,0.00,0.01,0.00,0,0.01,2500,0,100,2026-04-02,5.12695,-0.00052,0.00003,-0.06294,0.00025,-0.00000
2,AAPL260402C00120000,AAPL,2026-04-02,120.0,call,0.00,135.70,134.00,165,137.40,165,0,0,2026-04-02,0.01488,1.00000,0.00000,-0.01197,0.00000,0.00329
3,AAPL260402P00120000,AAPL,2026-04-02,120.0,put,0.00,0.01,0.00,0,0.01,2000,0,4,2026-04-02,4.61964,-0.00057,0.00003,-0.06205,0.00027,-0.00000
4,AAPL260402C00125000,AAPL,2026-04-02,125.0,call,128.89,130.60,128.85,162,132.35,160,1,1,2026-04-02,0.01488,1.00000,0.00000,-0.01246,0.00000,0.00342


## Filtered tickers have options on the universe end date

Loop over `FILTERED_TICKERS` and confirm each has a 2026-04-02 options file.


In [16]:
missing = []
for ticker in tqdm(FILTERED_TICKERS):
    path = Path(f"data/options_alpha/{ticker}/{ticker}_options_2026-04-02.csv")
    if not path.exists():
        missing.append(ticker)
print(f"missing on 2026-04-02: {missing}")

100%|██████████| 59/59 [00:00<00:00, 28424.53it/s]

missing on 2026-04-02: []
